# Simple Exponential Smoothing (SES) as a Weighted Average

Naïve uses **only the last value**; simple average uses **all values equally**. SES sits in between: it uses **all** past values but gives **more weight to recent ones**, with the weights fading away (but never quite reaching zero) as we go back in time.

## 1. The core idea

- SES can be written as a **weighted average of all past observations**, where the weights are **largest for the most recent data point** and **decline smoothly and exponentially** as we go back in time — **never fully dropping to zero**.
- This decaying-weight structure is exactly what gives the method the name **“exponential smoothing”**.

### Weights given to each past observation

| Observation | Weight |
|---|---|
| today, $y_t$ | $\alpha$ |
| 1 step back, $y_{t-1}$ | $\alpha(1-\alpha)$ |
| 2 steps back, $y_{t-2}$ | $\alpha(1-\alpha)^2$ |
| 3 steps back, $y_{t-3}$ | $\alpha(1-\alpha)^3$ |
| $\vdots$ | $\vdots$ |

Because $0 < \alpha < 1$, each factor of $(1-\alpha)$ shrinks the weight → the further back an observation, the smaller its influence (the decreasing-weight curve in the slide).

## 2. Exponential weighting — the forecast equation

The one-step-ahead forecast is:

$$\hat{y}_{t+1\mid t} \;=\; \alpha\,y_t \;+\; \alpha(1-\alpha)\,y_{t-1} \;+\; \alpha(1-\alpha)^2\,y_{t-2} \;+\; \cdots$$

Or compactly, the **general weighted-average expression**:

$$\hat{y}_{t+1\mid t} \;=\; \sum_{j=0}^{t-1} \alpha(1-\alpha)^{j}\, y_{t-j}$$

- **Every past observation contributes** to the forecast, with decreasing influence.
- **Only one parameter**, $\alpha \in (0,1)$, determines how quickly the weights decay.

**Worked example ($\alpha = 0.8$):** weights are $0.8,\; 0.8(0.2)=0.16,\; 0.8(0.2)^2=0.032,\;\dots$ — each weight is one-fifth of the previous, so older points fade fast.

## 3. The role of $\alpha$ (the smoothing parameter)

$\alpha$ controls **how responsive** the model is:

- **Larger $\alpha$** (near 1) → weights decay quickly → forecasts respond **more quickly** to recent changes.
- **Smaller $\alpha$** (near 0) → weights decay slowly → forecasts are **smoother** and rely more on the long history of past values.

### Extreme cases

| $\alpha$ | Behaviour |
|---|---|
| $\alpha = 1$ | Forecast = last observation → **naïve method** (all weight on $y_t$) |
| $\alpha \approx 0$ | Very slow-moving average → **long-term memory**, barely reacts to new data |

## 4. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4)

## 5. The weights decay exponentially

Let's actually look at $\alpha(1-\alpha)^j$ for a few values of $\alpha$. Note how the weights shrink as we go back in time, yet **never hit zero** — and how they sum to (almost) 1.

In [ ]:
j = np.arange(0, 12)            # 0 = today, 1 = one step back, ...

for alpha in [0.2, 0.5, 0.8]:
    weights = alpha * (1 - alpha) ** j
    plt.plot(j, weights, marker="o", label=f"α = {alpha}")

plt.gca().invert_xaxis()        # so 'today' sits on the right, like the slide
plt.xlabel("steps back in time  (0 = today)")
plt.ylabel("weight  α(1−α)ʲ")
plt.title("Exponentially decaying weights")
plt.legend()
plt.show()

In [ ]:
# The weights form a geometric series that sums to 1 (over infinite history)
for alpha in [0.2, 0.5, 0.8]:
    w = alpha * (1 - alpha) ** np.arange(0, 200)
    print(f"α = {alpha}:  first 4 weights = {np.round(w[:4], 4)} ...  sum = {w.sum():.4f}")

## 6. SES from the weighted-average formula (by hand)

We compute $\hat{y}_{t+1\mid t} = \sum_{j} \alpha(1-\alpha)^j\,y_{t-j}$ directly, summing over the available history (most recent first).

In [ ]:
# Our familiar 6-month series
y = np.array([5, 4, 7, 11, 9, 6], dtype=float)

def ses_forecast(y, alpha):
    """One-step-ahead forecast via the weighted-average expansion."""
    y_recent_first = y[::-1]                     # y_t, y_{t-1}, y_{t-2}, ...
    j = np.arange(len(y))
    weights = alpha * (1 - alpha) ** j
    # weights don't sum exactly to 1 over a finite series -> renormalise
    return np.sum(weights * y_recent_first) / weights.sum()

for alpha in [0.2, 0.5, 0.8, 1.0]:
    print(f"α = {alpha}:  forecast for month 7 = {ses_forecast(y, alpha):.3f}")

Notice $\alpha = 1$ reproduces the **naïve** forecast (= last value, 6), confirming the extreme case. Smaller $\alpha$ pulls the forecast toward the overall average.

## 7. The equivalent recursive form

The infinite weighted sum is usually computed with a simple recursion that gives **identical** results:

$$\hat{y}_{t+1\mid t} = \alpha\,y_t + (1-\alpha)\,\hat{y}_{t\mid t-1}$$

i.e. *new forecast = $\alpha \times$ (latest actual) + $(1-\alpha) \times$ (previous forecast)*. This is why SES needs to store only one number, not the whole history.

In [ ]:
def ses_recursive(y, alpha):
    level = y[0]                     # initialise with the first observation
    fitted = [level]
    for t in range(1, len(y)):
        level = alpha * y[t] + (1 - alpha) * level
        fitted.append(level)
    next_forecast = level            # forecast for the step after the last point
    return np.array(fitted), next_forecast

alpha = 0.5
fitted, fc = ses_recursive(y, alpha)

pd.DataFrame({"month": range(1, 7), "actual": y, "smoothed_level": np.round(fitted, 3)})

In [ ]:
# Visualise how different alphas track the data
plt.plot(range(1, 7), y, "ko-", label="actual")
for alpha in [0.2, 0.8]:
    fitted, _ = ses_recursive(y, alpha)
    plt.plot(range(1, 7), fitted, marker="s", label=f"SES α={alpha}")
plt.xlabel("month"); plt.ylabel("value")
plt.title("Smaller α = smoother;  larger α = follows recent data")
plt.legend(); plt.show()

## 8. Using statsmodels (the real tool)

In practice you'd let `statsmodels` fit SES (and even pick $\alpha$ by minimising SSE) rather than coding the recursion yourself.

In [ ]:
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

model = SimpleExpSmoothing(y, initialization_method="heuristic").fit(smoothing_level=0.5, optimized=False)
print("forecast (α=0.5):", model.forecast(1))

# Let statsmodels choose the best alpha automatically
auto = SimpleExpSmoothing(y, initialization_method="estimated").fit()
print("optimised α   :", round(auto.model.params["smoothing_level"], 3))
print("forecast      :", auto.forecast(1))

## 9. Summary

- **SES = weighted average of *all* past observations**, weights $\alpha(1-\alpha)^j$ that decay exponentially and never reach zero.
- **One parameter $\alpha \in (0,1)$** controls the decay / responsiveness.
  - large $\alpha$ → reacts fast to recent changes; $\alpha = 1$ → naïve method.
  - small $\alpha$ → smooth, long memory; $\alpha \approx 0$ → near-constant slow average.
- The weighted-sum and the recursion $\hat{y}_{t+1} = \alpha y_t + (1-\alpha)\hat{y}_t$ are **equivalent**.
- Best for series with **no clear trend or seasonality** (those need Holt / Holt-Winters).